In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.ensemble import RandomForestRegressor

os.chdir('../')
%pwd

'/media/om/volume2/MLOPS/The Ultimate MLOPS Course/youtube_project/MLOPS_youtube_CTA_Project'

In [2]:
df= pd.read_csv("./data/raw/youtube_10000_videos.csv")
df.head()

,category,channel_id,channel_name,subscriber_count,channel_view_count,channel_video_count,video_id,video_title,published_at,duration_seconds,view_count,like_count,comment_count,description,video_url
0,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,MOlaKBJ1nDA,प्रश्नावली 5.2 Class 10 Maths | NCERT Class 10...,2026-07-18T01:52:16Z,3988.0,427,21,3,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=MOlaKBJ1nDA
1,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,uXRH_uqCOXI,प्रश्नावली 5.3 Class 10 Maths | NCERT Class 10...,2026-07-19T02:22:39Z,3606.0,528,32,2,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=uXRH_uqCOXI
2,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,q4Q4dg6JDE0,प्रश्नावली 5.3 Class 10 Maths | NCERT Class 10...,2026-07-22T01:57:46Z,3985.0,520,23,2,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=q4Q4dg6JDE0
3,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,UTrN2vvjau8,Class 10 Maths | निर्देशांक ज्यामिति (Coordina...,2026-07-23T01:56:33Z,4655.0,566,27,0,Class 10 Maths | Coordinate Geometry Part 01 |...,https://www.youtube.com/watch?v=UTrN2vvjau8
4,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,22S8ptzHl-g,Class 10 Maths | निर्देशांक ज्यामिति (Coordina...,2026-07-24T02:38:43Z,2185.0,355,23,3,Class 10 Maths | Coordinate Geometry Part 01 |...,https://www.youtube.com/watch?v=22S8ptzHl-g


In [ ]:
required_columns=['category','subscriber_count','channel_view_count','duration_seconds', 'view_count']
df1=df[required_columns]
df1.head()

,category,subscriber_count,channel_view_count,duration_seconds,view_count
0,Education,167000,5169057,3988.0,427
1,Education,167000,5169057,3606.0,528
2,Education,167000,5169057,3985.0,520
3,Education,167000,5169057,4655.0,566
4,Education,167000,5169057,2185.0,355


In [4]:
from src.utils.remove_null_values import remove_null
# from src.utils.remove_outliers import remove_outliers
df2=remove_null(df1)
columns_list=list(df2.columns)
columns_list.remove('category')
print(columns_list)
# df2= remove_outliers(df2,columns_list)
df2

Total null values removed:  6
percent of null values removed:  0.06027122049221497
['subscriber_count', 'channel_view_count', 'duration_seconds', 'view_count']


,category,subscriber_count,channel_view_count,duration_seconds,view_count
0,Education,167000,5169057,3988.0,427
1,Education,167000,5169057,3606.0,528
2,Education,167000,5169057,3985.0,520
3,Education,167000,5169057,4655.0,566
4,Education,167000,5169057,2185.0,355
...,...,...,...,...,...
9950,Travel,127000,18680974,3719.0,66816
9951,Travel,127000,18680974,4181.0,232027
9952,Travel,127000,18680974,3743.0,85656
9953,Travel,127000,18680974,22.0,147597


In [5]:
from src.utils.combined_pipeline import combined_transform,model_with_combined_processor
numeric_features = ['subscriber_count','channel_view_count','duration_seconds']
categorical_features = ["category"]
combined_processor_pipeline= combined_transform(numeric_features,categorical_features)
model_pipeline = model_with_combined_processor(combined_processor_pipeline,RandomForestRegressor(n_estimators=1000,random_state=42,n_jobs=-1,max_depth=30,min_samples_leaf=10))

In [6]:
X=df2.drop(columns=["view_count"])
y= np.log1p(df2["view_count"])

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

In [7]:
from sklearn.metrics import r2_score, mean_squared_error

model_pipeline.fit(X_train, y_train)

predictions = model_pipeline.predict(X_test)

y_pred= model_pipeline.predict(X_test)
r2= r2_score(y_test,y_pred)
r2

0.7687906174540813

In [8]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import mlflow
import mlflow.sklearn
import dagshub

from src.utils.combined_pipeline import combined_transform,model_with_combined_processor
from src.utils.yaml_loader import yaml_loader

data= yaml_loader("./params.yaml")
params= data["params"]

results=[]

dagshub.init(repo_owner='AIforeverything', repo_name='MLOPS_youtube_CTA_Project', mlflow=True)

with mlflow.start_run(run_name='EXP-4_RandomForestRegressor') as run:
    
    numeric_features = ['subscriber_count','channel_view_count','duration_seconds']
    categorical_features = ["category"]
    combined_processor_pipeline= combined_transform(numeric_features,categorical_features)
    model_pipeline = model_with_combined_processor(combined_processor_pipeline,RandomForestRegressor(n_estimators=1000,random_state=42,n_jobs=-1,max_depth=30,min_samples_leaf=10))
    
        
    mlflow.log_param('algorithm', 'RandomForestRegressor')
    mlflow.log_param('cv_fold',5)
    mlflow.log_param(
                "scoring",
                "neg_mean_squared_error"
            )

    # --------------------------------------------------
    # Train GridSearchCV
    # -------------------------------------------------
    model_pipeline.fit(
        X_train,
        y_train)

    
    # --------------------------------------------------
    # Test prediction
    # -------------------------------------------------
    y_pred = model_pipeline.predict(X_test)
    # --------------------------------------------------
    # Test metrics
    # -------------------------------------------------
    mse = mean_squared_error(
        y_test,
        y_pred)
    
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(
        y_test,
        y_pred)
    
    r2 = r2_score(
        y_test,
        y_pred)
    
    # --------------------------------------------------
    # Log test metrics
    # -------------------------------------------------
    mlflow.log_metric(
        "test_mse",
        mse)
    
    mlflow.log_metric(
        "test_rmse",
        rmse)
    
    mlflow.log_metric(
        "test_mae",
        mae)
    
    mlflow.log_metric(
        "test_r2",
        r2)
    
    # --------------------------------------------------
    # Log best model
    # -------------------------------------------------
    mlflow.sklearn.log_model(
        model_pipeline,
        name="model",
        skops_trusted_types=["numpy.dtype",'xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor'])
    
    # --------------------------------------------------
    # Store result
    # -------------------------------------------------
    results.append({
        "model": 'RandomForestRegressor',
        "test_mse": mse,
        "test_rmse": rmse,
        "test_mae": mae,
        "test_r2": r2,
    })
    print("Parameters:")
    print(f"Test MSE: {mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test MAE : {mae:.4f}")
    print(f"Test R²  : {r2:.4f}")

/home/om/Desktop/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as AIforeverything

Initialized MLflow to track repo "AIforeverything/MLOPS_youtube_CTA_Project"

Repository AIforeverything/MLOPS_youtube_CTA_Project initialized!

Parameters:
Test MSE: 1.5585
Test RMSE: 1.2484
Test MAE : 0.9069
Test R²  : 0.7688
🏃 View run EXP-4_RandomForestRegressor at: https://dagshub.com/AIforeverything/MLOPS_youtube_CTA_Project.mlflow/#/experiments/0/runs/cdfdcaecab1a454cb3923e5c0bce4b6a
🧪 View experiment at: https://dagshub.com/AIforeverything/MLOPS_youtube_CTA_Project.mlflow/#/experiments/0
